# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and available online:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}  License: {dataset.metadata.license}")
print(f"Citation: {getattr(dataset.metadata, 'cite_as', getattr(dataset.metadata, 'citeAs', None))}")

## 2. Data Overview
Review available record sets and fields. All record sets, fields, and columns are referenced via their `@id` as per Croissant schema best practices.

In [ ]:
# List all record sets
record_sets = dataset.metadata.record_set  # Typically a list of record set objects
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs.get('schema:name', ''))}")

# Show fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord set '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"Fields:")
    for field in fields:
        print(f"  - {field['@id']}: {field.get('name', field.get('schema:name', ''))}")
        columns = field.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            print(f"      Column: {col['@id']} ({col.get('name', col.get('schema:name', ''))})")

## 3. Data Extraction
Load data from one or more record sets into Pandas `DataFrame`s using `@id` references.

*Replace the record set and field IDs below with the desired ones from the above overview.*

In [ ]:
# Choose one or more record sets by their '@id'
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {list(df.columns)}")
    else:
        print(f"  No records found for {record_set_id}.")

# Preview a DataFrame if any data was loaded
if dataframes:
    sample_rs = list(dataframes.keys())[0]
    print(f"\nSample rows from {sample_rs}:")
    display(dataframes[sample_rs].head())

## 4. Exploratory Data Analysis (EDA)
Below we demonstrate numeric filtering, normalization, and grouping for one selected record set and field using IDs. If no numeric data is present, adapt as needed.

In [ ]:
# Select a record set '@id' and relevant field/column '@id'
record_set_id = list(dataframes.keys())[0]  # The '@id' of a loaded record set
df = dataframes[record_set_id]

# Find a numeric field (e.g., 'MW' for molecular weight, or 'Coverage', etc.; update as appropriate)
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col].dropna().astype(str).str.replace(',', '.'))]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    if not pd.api.types.is_numeric_dtype(df[numeric_field]):
        # Try to convert if not numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    print(f"Using numeric field: {numeric_field}")
else:
    print("No obvious numeric field found.")
    numeric_field = None

# Filtering and normalization if we have a numeric field
if numeric_field is not None:
    threshold = df[numeric_field].mean() if df[numeric_field].mean() != 0 else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    group_fields = [col for col in df.columns if col != numeric_field]
    group_field = None
    for col in group_fields:
        if df[col].nunique() < min(15, len(df)//10) and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("Skip numeric EDA as no numeric field was detected.")

## 5. Visualization
Visualize data distributions or relationships for fields in the dataset.

In [ ]:
# Basic visualization of the chosen numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to load the FAIR² dataset, discover its structure using Croissant schema IDs, efficiently extract records, and perform basic EDA. The dataset lends itself well to studies of protein expression and mass spectrometry analysis. For more advanced statistical or ML analysis, refer to the field and column `@id` mappings provided by the Croissant schema.